In [8]:
from dotenv import load_dotenv

load_dotenv()

import os

model = os.getenv('MODEL_NAME', 'glm-5.2')

In [9]:
from crewai import LLM, Agent, Task, Crew, Process
import asyncio
import nest_asyncio

nest_asyncio.apply()

STRATEGY_TASK = """
请分析以下社交媒体主题：
主题：{topic}
品牌：{brand}
平台：{platforms}

请制定以下内容：
1. 核心传播信息
2. 目标受众
3. 情感吸引点（Emotional Hook）
4. 5 个相关的话题标签（Hashtags）
"""

WRITING_TASK = """
请为以下平台分别创作社交媒体内容：
主题：{topic}
品牌：{brand}
平台：{platforms}
请分别生成：
1. Twitter/X
- 2 个推文版本（每条不超过 280 个字符）
- 1 个推文线程（Thread）的开场内容
2. 小红书
- 一篇笔记（300～500 字）
- 包含吸引人的标题
- 开头能够快速抓住用户注意力
- 内容真实、有分享感，必须加入 Emoji
- 结尾引导用户评论或收藏
- 附带 10～15 个相关话题标签
3. 抖音
- 一段短视频文案
- 包括：
    * 视频标题
    * 前 3 秒吸引人的开场（Hook）
    * 视频口播文案（约 150～250 字）
    * 结尾互动引导（点赞、评论、关注等）
    * 5～8 个相关话题标签

请确保内容符合各平台特点：
- Twitter/X：简洁、有冲击力、易传播。
- 小红书：生活化、真实、有种草分享感，适合图文笔记。
- 抖音：节奏快、吸引力强，适合短视频传播。
"""

def generate_social_content(topic: str, brand: str, platforms: list[str]) -> str:
    llm = LLM(model=model, temperature=0.7)

    strategist = Agent(
        role="社交媒体策略师",
        goal="分析主题，并为每个平台制定核心信息、目标受众和内容语调",
        backstory="屡获殊荣的社交媒体策略专家，曾帮助 50 多个品牌账号增长至 10 万+ 粉丝。",
        llm=llm,
        verbose=True,
    )

    writer = Agent(
        role="社交媒体文案撰写人",
        goal="创作具有吸引力、符合各平台特点的内容，以提升用户互动率",
        backstory="擅长打造爆款内容的创作者，精通不同社交平台的文案风格、话题标签和吸引人的开头。",
        llm=llm,
        verbose=True,
    )

    strategy_task = Task(
        description=STRATEGY_TASK.format(topic=topic, brand=brand or "通用品牌", platforms=('，'.join(platforms))),
        agent=strategist,
        expected_output="内容策略，包括核心信息、目标受众、情感吸引点以及话题标签。",
    )

    writing_task = Task(
        description=WRITING_TASK.format(topic=topic, brand=brand or "通用品牌", platforms=('，'.join(platforms))),
        agent=writer,
        expected_output="针对所有指定平台生成符合平台特点的社交媒体内容。",
        context=[strategy_task],
    )

    crew = Crew(
        agents=[strategist, writer],
        tasks=[strategy_task, writing_task],
        process=Process.sequential,
        verbose=True,
    )

    result = asyncio.run(crew.kickoff_async())
    return str(result)

In [ ]:
result = generate_social_content(topic='男士面霜', brand='欧莱雅', platforms=['小红书'])

print(result)

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: 9de7775a-deb5-4f60-901c-953fba148912                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name:                                                                       │
│  请分析以下社交媒体主题：              

╭─────────────────────────────── Tracing Status ───────────────────────────────╮
│                                                                              │
│  Info: Tracing is disabled.                                                  │
│                                                                              │
│  To enable tracing, do any one of these:                                     │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯
